In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhatduc2425/slake-rag/slake_gated_fusion.pth
/kaggle/input/datasets/nhatduc2425/slake-rag/slake_metadata.pt
/kaggle/input/datasets/nhatduc2425/slake-rag/slake_fusion.index


In [2]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 57.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [3]:
import torch
import time
import evaluate
import numpy as np
import gc
import os
import random
import urllib.request
import zipfile
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import Image, ImageFile
from transformers import pipeline

ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- 1. ĐƯỜNG DẪN TRÊN KAGGLE ---
IMG_DIR = "/kaggle/working/SLAKE_imgs"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# --- TỰ ĐỘNG QUẢN LÝ ẢNH SLAKE TRÊN KAGGLE ---
def ensureImagesExist():
    if not os.path.exists(IMG_DIR) or len(os.listdir(IMG_DIR)) == 0:
        print("⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE về /kaggle/working/...")
        urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
        print("📦 Đang giải nén ảnh...")
        with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
            zipRef.extractall(IMG_DIR)
        os.remove("imgs.zip")
        print("✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!")

def getImagePath(imgName):
    if not imgName: return None
    path1 = os.path.join(IMG_DIR, imgName)
    path2 = os.path.join(IMG_DIR, "imgs", imgName)
    if os.path.exists(path1): return path1
    if os.path.exists(path2): return path2
    return None

# --- MÔ HÌNH SUY LUẬN (ZERO-SHOT) ---
class ZeroShotSlmGenerator:
    def __init__(self):
        print("\n[Hạ tầng] Đang tải Qwen2-VL-7B-Instruct (Zero-Shot Mode) và phân bổ lên cấu hình Multi-GPU Kaggle...")
        self.pipe = pipeline(
            "image-text-to-text",
            model="Qwen/Qwen2-VL-7B-Instruct", 
            device_map="auto",
            model_kwargs={"torch_dtype": torch.bfloat16} 
        )
        print("✅ Đã nạp thành công mô hình cấu hình 7B!")

    def optimizeImage(self, pilImg, maxRes=336):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        pilImg.thumbnail((maxRes, maxRes), Image.Resampling.LANCZOS)
        return pilImg

    def generate(self, testImg, testQuest):
        contentBlock = []
        
        # System prompt được tinh chỉnh cho Zero-shot (Không có RAG)
        systemInstruction = (
            "<System>\n"
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based EXCLUSIVELY on the provided medical image.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Target Case Image] and the [Question].\n"
            "2. Provide a concise and definitive answer (1-3 words if possible).\n"
            "</System>\n\n"
            "<User>\n"
            "[Target Case Image]\n"
        )
        contentBlock.append({"type": "text", "text": systemInstruction})
        
        optimizedTestImg = self.optimizeImage(testImg)
        contentBlock.append({"type": "image", "image": optimizedTestImg})
        
        targetInstruction = (
            f"\nCurrent Question: {testQuest}\n\n"
            "Based on the visual evidence, what is the correct answer for the current case?\n"
            "</User>\n\n"
            "<Assistant>\n"
            "Final Answer: "
        )
        contentBlock.append({"type": "text", "text": targetInstruction})
        
        messages = [{"role": "user", "content": contentBlock}]
        outputs = self.pipe(
            messages, 
            max_new_tokens=15, 
            generate_kwargs={"do_sample": False, "max_length": None}
        )
        
        rawResponse = outputs[0]['generated_text']
        finalText = rawResponse[-1]['content'] if isinstance(rawResponse, list) else rawResponse
        return finalText.replace("</Assistant>", "").strip()

# --- HÀM THỰC THI CHÍNH ---
def runKaggleZeroShot7bBenchmark(sampleSize=500):
    print("1. Khởi tạo và kiểm tra dữ liệu hình ảnh...")
    ensureImagesExist()
    
    print("2. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("3. Tải mô hình suy luận...")
    slm = ZeroShotSlmGenerator()
    
    rawTestDS = load_dataset("BoKelvin/SLAKE", split="test")
    englishTestDS = [item for item in rawTestDS if item.get('q_lang') != 'zh']
    
    totalIndices = list(range(len(englishTestDS)))
    sampledIndices = random.sample(totalIndices, min(sampleSize, len(englishTestDS)))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY ZERO-SHOT BASELINE (7B - KHÔNG RAG) TRÊN {len(sampledIndices)} MẪU KAGGLE...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()
    skippedCount = 0

    for progressCount, idx in enumerate(sampledIndices):
        item = englishTestDS[idx]
        imgName = item.get('img_name')
        imgPath = getImagePath(imgName)
        
        if not imgPath: 
            skippedCount += 1
            continue
            
        startTimeSample = time.time()
        
        try:
            rawImg = Image.open(imgPath).convert("RGB")
            # Gọi hàm suy luận chỉ với ảnh và câu hỏi
            aiAnswerRaw = slm.generate(rawImg, str(item['question']))
        except Exception as e:
            print(f"⚠️ Lỗi ngầm tại mẫu {idx}: {e}")
            skippedCount += 1
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 50 == 0: 
            print(f"Tiến độ Kaggle (7B - Zero-Shot): {progressCount + 1}/{len(sampledIndices)} mẫu...")

    endTotalTime = time.time()
    
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM ĐỐI CHỨNG CÔNG BẰNG - ZERO-SHOT (QWEN-7B)\n" + "="*60)
    print(f"📌 Đã quét: {len(sampledIndices)} | Bỏ qua do lỗi ảnh: {skippedCount}")
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    if latencyList:
        print("\n[3. Computational Efficiency Measurements]")
        print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
        print(f"  - Throughput (Tốc độ xử lý)             : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}       : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runKaggleZeroShot7bBenchmark(sampleSize=500)

1. Khởi tạo và kiểm tra dữ liệu hình ảnh...
⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE về /kaggle/working/...
📦 Đang giải nén ảnh...
✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!
2. Khởi tạo các thư viện đo lường (Metrics)...


3. Tải mô hình suy luận...

[Hạ tầng] Đang tải Qwen2-VL-7B-Instruct (Zero-Shot Mode) và phân bổ lên cấu hình Multi-GPU Kaggle...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

... (26 outputs left out)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  - BLEU-1   : 0.2255
  - BLEU-2   : 0.0896
  - BLEU-3   : 0.0426
  - ROUGE-L  : 0.3371
  - BERTScore: 0.9162

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 2.44 s/mẫu
  - Throughput (Tốc độ xử lý)             : 0.41 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0       : 8297.18 MB
  - Peak Memory Usage on GPU 1       : 8660.54 MB
